In [2]:
import xgboost as xgb

# Initialize XGBRegressor with GPU support
xgb_model = xgb.XGBRegressor(
    tree_method='gpu_hist',  # Use GPU for training
    predictor='gpu_predictor',  # Use GPU for predictions
    gpu_id=0,  # Use the first GPU
    objective='reg:squarederror',
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6
)

# Check if GPU is enabled
print(xgb_model.get_params())

{'objective': 'reg:squarederror', 'base_score': None, 'booster': None, 'colsample_bylevel': None, 'colsample_bynode': None, 'colsample_bytree': None, 'enable_categorical': False, 'gamma': None, 'gpu_id': 0, 'importance_type': None, 'interaction_constraints': None, 'learning_rate': 0.1, 'max_delta_step': None, 'max_depth': 6, 'min_child_weight': None, 'missing': nan, 'monotone_constraints': None, 'n_estimators': 100, 'n_jobs': None, 'num_parallel_tree': None, 'predictor': 'gpu_predictor', 'random_state': None, 'reg_alpha': None, 'reg_lambda': None, 'scale_pos_weight': None, 'subsample': None, 'tree_method': 'gpu_hist', 'validate_parameters': None, 'verbosity': None}


In [ ]:
import lightgbm as lgb
from catboost import CatBoostRegressor
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split

# Create a synthetic dataset
X, y = make_regression(n_samples=10000, n_features=20, random_state=42)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# LightGBM with GPU
print("Training LightGBM with GPU...")
lgb_params = {
    'device_type': 'gpu',  # Use GPU
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'num_leaves': 31,
    'learning_rate': 0.1,
    'verbose': 1
}
train_data = lgb.Dataset(X_train, label=y_train)
lgb_model = lgb.train(lgb_params, train_data, num_boost_round=100)
print("LightGBM training complete!")

# CatBoost with GPU
print("Training CatBoost with GPU...")
cat_model = CatBoostRegressor(
    task_type='GPU',  # Use GPU
    iterations=100,
    learning_rate=0.1,
    depth=6,
    verbose=200  # Show logs
)
cat_model.fit(X_train, y_train)
print("CatBoost training complete!")

Training LightGBM with GPU...
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 5100
[LightGBM] [Info] Number of data points in the train set: 8000, number of used features: 20
[LightGBM] [Info] Using GPU Device: NVIDIA GeForce RTX 3080 Ti, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 20 dense feature groups (0.15 MB) transferred to GPU in 0.002330 secs. 0 sparse feature groups
[LightGBM] [Info] Start training from score -1.823056
[LightGBM] [Debug] Trained a tree with leaves = 31 and depth = 6
[LightGBM] [Debug] Trained a tree with leaves = 31 and depth = 6
[LightGBM] [Debug] Trained a tree with leaves = 31 and depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 31 and depth = 7
[LightGBM] [Debug] Trained a tree with leaves = 31 and depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 31 and depth =

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import matplotlib.pyplot as plt
import time
import pickle
from sklearn.model_selection import train_test_split, RandomizedSearchCV, GridSearchCV, KFold, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import VotingRegressor, StackingRegressor
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
import optuna
import warnings
warnings.filterwarnings('ignore')
warnings.filterwarnings("ignore", message="numpy.dtype size changed")
# Load the dataset
df = pd.read_csv('bus_eta_standard_scaled.csv')

print("Data shape:", df.shape)

# Selecting features and target variable
X = df.drop(columns=['timestamp', 'eta_minutes'])  # Features
y = df['eta_minutes']  # Target

# Optional: Feature engineering
# You can add more engineered features here if needed

# Splitting the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training set shape:", X_train.shape)
print("Testing set shape:", X_test.shape)

# Function to evaluate and visualize model performance
def evaluate_model(model, X_test, y_test, model_name):
    # Make predictions
    y_pred = model.predict(X_test)
    
    # Calculate metrics
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100
    accuracy = 100 - mape
    
    print(f"--- {model_name} Performance Metrics ---")
    print(f'RMSE: {rmse:.4f}')
    print(f'MAE: {mae:.4f}')
    print(f'R² Score: {r2:.4f}')
    print(f"MAPE: {mape:.2f}%")
    print(f"Prediction Accuracy: {accuracy:.2f}%")
    
    # Create visualizations
    fig, axes = plt.subplots(2, 2, figsize=(20, 15))
    
    # Scatter plot of actual vs predicted
    axes[0, 0].scatter(y_test, y_pred, alpha=0.5)
    axes[0, 0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r', linestyle='--')
    axes[0, 0].set_xlabel("Actual ETA")
    axes[0, 0].set_ylabel("Predicted ETA")
    axes[0, 0].set_title(f"{model_name}: Actual vs. Predicted ETA")
    
    # Line plot of first 50 samples
    axes[0, 1].plot(y_test.values[:50], label="Actual", marker='o')
    axes[0, 1].plot(y_pred[:50], label="Predicted", marker='x')
    axes[0, 1].set_xlabel("Sample Index")
    axes[0, 1].set_ylabel("ETA (minutes)")
    axes[0, 1].set_title(f"{model_name}: Actual vs. Predicted ETA (First 50 Samples)")
    axes[0, 1].legend()
    
    # Histogram of errors
    errors = y_pred - y_test
    axes[1, 0].hist(errors, bins=30, edgecolor='black')
    axes[1, 0].set_xlabel("Prediction Error (Predicted - Actual)")
    axes[1, 0].set_ylabel("Frequency")
    axes[1, 0].set_title(f"{model_name}: Histogram of Prediction Errors")
    
    # Percentage error vs actual
    percentage_error = 100 * (y_pred - y_test) / y_test
    axes[1, 1].scatter(y_test, percentage_error, alpha=0.5)
    axes[1, 1].axhline(y=0, color='r', linestyle='--')
    axes[1, 1].set_xlabel("Actual ETA (minutes)")
    axes[1, 1].set_ylabel("Prediction Error (%)")
    axes[1, 1].set_title(f"{model_name}: Prediction Error vs. Actual ETA")
    
    plt.tight_layout()
    plt.show()
    
    return {
        'rmse': rmse,
        'mae': mae,
        'r2': r2,
        'mape': mape,
        'accuracy': accuracy,
        'predictions': y_pred
    }

# Step 1: Optimize LightGBM using Optuna
print("\n=== Optimizing LightGBM with Optuna ===")

def objective(trial):
    """Optuna objective function for LightGBM optimization"""
    param = {
        'objective': 'regression',
        'metric': 'rmse',
        'boosting_type': trial.suggest_categorical('boosting_type', ['gbdt', 'dart', 'goss']),
        'num_leaves': trial.suggest_int('num_leaves', 20, 3000),
        'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.3),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.1, 1.0),
        'min_child_samples': trial.suggest_int('min_child_samples', 1, 100),
        'max_depth': trial.suggest_int('max_depth', 3, 25),
        'lambda_l1': trial.suggest_float('lambda_l1', 1e-8, 10.0, log=True),
        'lambda_l2': trial.suggest_float('lambda_l2', 1e-8, 10.0, log=True),
        'min_split_gain': trial.suggest_float('min_split_gain', 1e-8, 10.0, log=True),
        'device_type': 'gpu',  # Use GPU acceleration
        'verbosity': -1  # Control verbosity here
    }
    
    # Add GOSS-specific or DART-specific parameters
    if param['boosting_type'] == 'goss':
        # Ensure top_rate + other_rate <= 1.0
        top_rate = trial.suggest_float('top_rate', 0.0, 0.5)  # Limit top_rate to 0.5
        other_rate = trial.suggest_float('other_rate', 0.0, 1.0 - top_rate)  # Limit other_rate to 1.0 - top_rate
        param['top_rate'] = top_rate
        param['other_rate'] = other_rate
        # Remove bagging parameters for GOSS
        param.pop('bagging_fraction', None)
        param.pop('bagging_freq', None)
    elif param['boosting_type'] == 'dart':
        param['drop_rate'] = trial.suggest_float('drop_rate', 0.0, 1.0)
        param['skip_drop'] = trial.suggest_float('skip_drop', 0.0, 1.0)
    else:
        # Add bagging parameters for gbdt
        param['bagging_fraction'] = trial.suggest_float('bagging_fraction', 0.1, 1.0)
        param['bagging_freq'] = trial.suggest_int('bagging_freq', 1, 10)
    
    # K-fold cross-validation
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    scores = []
    
    for train_idx, valid_idx in kf.split(X_train):
        X_train_fold, X_valid_fold = X_train.iloc[train_idx], X_train.iloc[valid_idx]
        y_train_fold, y_valid_fold = y_train.iloc[train_idx], y_train.iloc[valid_idx]
        
        dtrain = lgb.Dataset(X_train_fold, label=y_train_fold)
        dvalid = lgb.Dataset(X_valid_fold, label=y_valid_fold, reference=dtrain)
        
        model = lgb.train(
            param,
            dtrain,
            num_boost_round=10000,
            valid_sets=[dvalid],
            callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)]
        )
        
        y_pred = model.predict(X_valid_fold)
        score = np.sqrt(mean_squared_error(y_valid_fold, y_pred))
        scores.append(score)
    
    return np.mean(scores)

# Create Optuna study and optimize
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=100)  # Increase trials for better results

print("Best trial:")
trial = study.best_trial
print(f"  RMSE: {trial.value:.4f}")
print("  Params: ")
for key, value in trial.params.items():
    print(f"    {key}: {value}")

# Train LightGBM with the best params
print("\n=== Training LightGBM with Optimized Parameters ===")
best_params = trial.params.copy()
best_params['objective'] = 'regression'
best_params['metric'] = 'rmse'
best_params['device_type'] = 'gpu'
best_params['verbosity'] = -1

# Create datasets
lgb_train = lgb.Dataset(X_train, y_train)
lgb_eval = lgb.Dataset(X_test, y_test, reference=lgb_train)

# Train the model
start_time = time.time()
lgb_optimized = lgb.train(
    best_params,
    lgb_train,
    num_boost_round=10000,
    valid_sets=[lgb_eval],
    callbacks=[
        lgb.early_stopping(stopping_rounds=100, verbose=True),
        lgb.log_evaluation(period=100)
    ]
)
lgb_training_time = time.time() - start_time
print(f"LightGBM optimized model training time: {lgb_training_time:.2f} seconds")

# Evaluate the optimized LightGBM model
lgb_results = evaluate_model(lgb_optimized, X_test, y_test, "Optuna-Optimized LightGBM")

# Step 2: Train and optimize XGBoost
print("\n=== Training and Optimizing XGBoost ===")

def xgb_objective(trial):
    """Optuna objective function for XGBoost optimization"""
    param = {
        'objective': 'reg:squarederror',
        'tree_method': 'gpu_hist',  # Use GPU acceleration
        'predictor': 'gpu_predictor',
        'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.3),
        'max_depth': trial.suggest_int('max_depth', 3, 25),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 100),
        'gamma': trial.suggest_float('gamma', 1e-8, 1.0, log=True),
        'subsample': trial.suggest_float('subsample', 0.1, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.1, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'n_estimators': 10000,
        'verbosity': 0
    }
    
    # K-fold cross-validation
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    scores = []
    
    for train_idx, valid_idx in kf.split(X_train):
        X_train_fold, X_valid_fold = X_train.iloc[train_idx], X_train.iloc[valid_idx]
        y_train_fold, y_valid_fold = y_train.iloc[train_idx], y_train.iloc[valid_idx]
        
        model = XGBRegressor(**param, early_stopping_rounds=100, eval_metric='rmse')
        model.fit(
            X_train_fold, y_train_fold,
            eval_set=[(X_valid_fold, y_valid_fold)],
            verbose=False
        )
        
        y_pred = model.predict(X_valid_fold)
        score = np.sqrt(mean_squared_error(y_valid_fold, y_pred))
        scores.append(score)
    
    return np.mean(scores)

# Create Optuna study for XGBoost and optimize
xgb_study = optuna.create_study(direction='minimize')
xgb_study.optimize(xgb_objective, n_trials=50)  # Increase for better results

print("Best XGBoost trial:")
xgb_trial = xgb_study.best_trial
print(f"  RMSE: {xgb_trial.value:.4f}")
print("  Params: ")
for key, value in xgb_trial.params.items():
    print(f"    {key}: {value}")

# Train XGBoost with optimized parameters
xgb_best_params = xgb_trial.params.copy()
xgb_best_params['tree_method'] = 'gpu_hist'
xgb_best_params['predictor'] = 'gpu_predictor'
xgb_best_params['objective'] = 'reg:squarederror'
xgb_best_params['verbosity'] = 0

start_time = time.time()
xgb_model = XGBRegressor(**xgb_best_params, early_stopping_rounds=100)
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=100
)
xgb_training_time = time.time() - start_time
print(f"XGBoost optimized model training time: {xgb_training_time:.2f} seconds")

# Evaluate the optimized XGBoost model
xgb_results = evaluate_model(xgb_model, X_test, y_test, "Optuna-Optimized XGBoost")

# Step 3: Train and optimize CatBoost
print("\n=== Training and Optimizing CatBoost ===")

def catboost_objective(trial):
    """Optuna objective function for CatBoost optimization"""
    param = {
        'loss_function': 'RMSE',
        'eval_metric': 'RMSE',
        'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.3),
        'depth': trial.suggest_int('depth', 4, 16),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-8, 10.0, log=True),
        'random_strength': trial.suggest_float('random_strength', 1e-8, 10.0, log=True),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 10.0),
        'border_count': trial.suggest_int('border_count', 32, 255),
        'grow_policy': trial.suggest_categorical('grow_policy', ['SymmetricTree', 'Depthwise', 'Lossguide']),
        'task_type': 'GPU',  # Use GPU
        'verbose': 0
    }
    
    # K-fold cross-validation
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    scores = []
    
    for train_idx, valid_idx in kf.split(X_train):
        X_train_fold, X_valid_fold = X_train.iloc[train_idx], X_train.iloc[valid_idx]
        y_train_fold, y_valid_fold = y_train.iloc[train_idx], y_train.iloc[valid_idx]
        
        model = CatBoostRegressor(**param, iterations=10000, early_stopping_rounds=100)
        model.fit(
            X_train_fold, y_train_fold,
            eval_set=[(X_valid_fold, y_valid_fold)],
            verbose=False
        )
        
        y_pred = model.predict(X_valid_fold)
        score = np.sqrt(mean_squared_error(y_valid_fold, y_pred))
        scores.append(score)
    
    return np.mean(scores)

# Create Optuna study for CatBoost and optimize
cat_study = optuna.create_study(direction='minimize')
cat_study.optimize(catboost_objective, n_trials=50)  # Increase for better results

print("Best CatBoost trial:")
cat_trial = cat_study.best_trial
print(f"  RMSE: {cat_trial.value:.4f}")
print("  Params: ")
for key, value in cat_trial.params.items():
    print(f"    {key}: {value}")

# Train CatBoost with optimized parameters
cat_best_params = cat_trial.params.copy()
cat_best_params['task_type'] = 'GPU'
cat_best_params['verbose'] = 100

start_time = time.time()
cat_model = CatBoostRegressor(**cat_best_params, iterations=10000, early_stopping_rounds=100)
cat_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=100
)
cat_training_time = time.time() - start_time
print(f"CatBoost optimized model training time: {cat_training_time:.2f} seconds")

# Evaluate the optimized CatBoost model
cat_results = evaluate_model(cat_model, X_test, y_test, "Optuna-Optimized CatBoost")

# Step 4: Create an ensemble of models
print("\n=== Building Model Ensemble ===")

# Stacking Regressor
base_models = [
    ('lgb', lgb.LGBMRegressor(**{k: v for k, v in best_params.items() if k != 'device_type'}, device='gpu')),
    ('xgb', XGBRegressor(**xgb_best_params)),
    ('cat', CatBoostRegressor(**cat_best_params, iterations=cat_model.best_iteration_))
]

# Add some linear models to improve robustness
base_models.extend([
    ('ridge', Ridge(alpha=1.0)),
    ('elastic', ElasticNet(alpha=0.1, l1_ratio=0.5))
])

# Create meta-learner
meta_learner = Ridge()

# Create and train the stacking model
start_time = time.time()
stacking_model = StackingRegressor(
    estimators=base_models,
    final_estimator=meta_learner,
    cv=5,
    n_jobs=1
)

print("Training ensemble model...")
stacking_model.fit(X_train, y_train)
ensemble_training_time = time.time() - start_time
print(f"Ensemble model training time: {ensemble_training_time:.2f} seconds")

# Evaluate the ensemble model
ensemble_results = evaluate_model(stacking_model, X_test, y_test, "Stacking Ensemble Model")

# Step 5: Create a weighted ensemble for final predictions
print("\n=== Creating Weighted Ensemble ===")

# Voting Regressor with best weights
models = [
    ('lgb', lgb.LGBMRegressor(**{k: v for k, v in best_params.items() if k != 'device_type'}, device='gpu')),
    ('xgb', XGBRegressor(**xgb_best_params)),
    ('cat', CatBoostRegressor(**cat_best_params, iterations=cat_model.best_iteration_))
]

# Find optimal weights through cross-validation
def find_optimal_weights():
    from scipy.optimize import minimize
    
    # Function to minimize
    def objective_func(weights):
        # Normalize weights to sum to 1
        weights = weights / np.sum(weights)
        
        predictions = np.zeros(len(y_test))
        for i, (_, model) in enumerate(models):
            if isinstance(model, lgb.LGBMRegressor):
                model_copy = model.fit(X_train, y_train)
                predictions += weights[i] * model_copy.predict(X_test)
            else:
                model_copy = model.fit(X_train, y_train)
                predictions += weights[i] * model_copy.predict(X_test)
        
        return np.sqrt(mean_squared_error(y_test, predictions))
    
    # Initial weights: equal weighting
    initial_weights = np.ones(len(models)) / len(models)
    
    # Constraint: weights sum to 1
    constraints = ({'type': 'eq', 'fun': lambda w: np.sum(w) - 1})
    
    # Bounds: all weights between 0 and 1
    bounds = [(0, 1) for _ in range(len(models))]
    
    # Minimize the objective function
    result = minimize(objective_func, initial_weights, method='SLSQP', 
                      bounds=bounds, constraints=constraints)
    
    return result.x / np.sum(result.x)  # Normalize weights

# Find optimal weights
optimal_weights = find_optimal_weights()
print(f"Optimal weights: {optimal_weights}")

# Create and train a weighted voting regressor
voting_model = VotingRegressor(estimators=models, weights=optimal_weights)
voting_model.fit(X_train, y_train)

# Evaluate the weighted voting model
voting_results = evaluate_model(voting_model, X_test, y_test, "Weighted Voting Ensemble")

# Step 6: Compare all models and choose the best
print("\n=== Final Model Comparison ===")

models_comparison = {
    "LightGBM": lgb_results,
    "XGBoost": xgb_results,
    "CatBoost": cat_results,
    "Stacking Ensemble": ensemble_results,
    "Weighted Voting": voting_results
}

# Display comparison table
print(f"{'Model':<20} {'RMSE':<10} {'R²':<10} {'MAPE':<10} {'Accuracy':<10}")
print("-" * 60)

for model_name, results in models_comparison.items():
    print(f"{model_name:<20} {results['rmse']:<10.4f} {results['r2']:<10.4f} "
          f"{results['mape']:<10.2f} {results['accuracy']:<10.2f}")

# Find the best model based on RMSE
best_model_name = min(models_comparison.items(), key=lambda x: x[1]['rmse'])[0]
print(f"\nBest model based on RMSE: {best_model_name}")

# Save all models
print("\n=== Saving Models ===")

# Function to determine which model to save based on name
def save_model(model_name):
    if model_name == "LightGBM":
        lgb_optimized.save_model('lightgbm_ultimate_model.txt')
        return 'lightgbm_ultimate_model.txt'
    elif model_name == "XGBoost":
        xgb_model.save_model('xgboost_ultimate_model.json')
        return 'xgboost_ultimate_model.json'
    elif model_name == "CatBoost":
        cat_model.save_model('catboost_ultimate_model.cbm')
        return 'catboost_ultimate_model.cbm'
    elif model_name == "Stacking Ensemble":
        with open('stacking_ensemble_model.pkl', 'wb') as file:
            pickle.dump(stacking_model, file)
        return 'stacking_ensemble_model.pkl'
    elif model_name == "Weighted Voting":
        with open('voting_ensemble_model.pkl', 'wb') as file:
            pickle.dump(voting_model, file)
        return 'voting_ensemble_model.pkl'

# Save the best model
best_model_path = save_model(best_model_name)
print(f"Best model saved as: {best_model_path}")

# Save all models for potential use
for model_name in models_comparison.keys():
    if model_name != best_model_name:
        model_path = save_model(model_name)
        print(f"{model_name} saved as: {model_path}")

# Step 7: Feature importance analysis for the best model
print("\n=== Feature Importance Analysis ===")

def plot_feature_importance(model_name):
    plt.figure(figsize=(15, 10))
    
    if model_name == "LightGBM":
        lgb.plot_importance(lgb_optimized, max_num_features=20)
        plt.title('Feature Importance (LightGBM)')
    elif model_name == "XGBoost":
        xgb_importance = xgb_model.get_booster().get_score(importance_type='gain')
        importance_df = pd.DataFrame({'Feature': list(xgb_importance.keys()),
                                      'Importance': list(xgb_importance.values())})
        importance_df = importance_df.sort_values('Importance', ascending=False).head(20)
        plt.barh(importance_df['Feature'], importance_df['Importance'])
        plt.title('Feature Importance (XGBoost)')
    elif model_name == "CatBoost":
        cat_importance = cat_model.get_feature_importance()
        feature_names = X_train.columns
        importance_df = pd.DataFrame({'Feature': feature_names,
                                      'Importance': cat_importance})
        importance_df = importance_df.sort_values('Importance', ascending=False).head(20)
        plt.barh(importance_df['Feature'], importance_df['Importance'])
        plt.title('Feature Importance (CatBoost)')
    else:
        # For ensemble models, use LightGBM's importance as a proxy
        lgb.plot_importance(lgb_optimized, max_num_features=20)
        plt.title(f'Feature Importance (Proxy from LightGBM for {model_name})')
    
    plt.tight_layout()
    plt.show()

# Plot feature importance for the best model
plot_feature_importance(best_model_name)

# Step 8: Learning curve analysis
print("\n=== Learning Curve Analysis ===")

def plot_learning_curve(model, X, y, title="Learning Curve", ylim=None, cv=5,
                        n_jobs=-1, train_sizes=np.linspace(.1, 1.0, 5)):
    plt.figure(figsize=(10, 6))
    plt.title(title)
    if ylim is not None:
        plt.ylim(*ylim)
    plt.xlabel("Training examples")
    plt.ylabel("Score (negative RMSE)")
    
    train_sizes, train_scores, test_scores = learning_curve(
        model, X, y, cv=cv, n_jobs=n_jobs, train_sizes=train_sizes,
        scoring='neg_root_mean_squared_error')
    
    train_scores_mean = np.mean(train_scores, axis=1)
    train_scores_std = np.std(train_scores, axis=1)
    test_scores_mean = np.mean(test_scores, axis=1)
    test_scores_std = np.std(test_scores, axis=1)
    
    plt.grid()
    plt.fill_between(train_sizes, train_scores_mean - train_scores_std,
                     train_scores_mean + train_scores_std, alpha=0.1, color="r")
    plt.fill_between(train_sizes, test_scores_mean - test_scores_std,
                     test_scores_mean + test_scores_std, alpha=0.1, color="g")
    plt.plot(train_sizes, train_scores_mean, 'o-', color="r", label="Training score")
    plt.plot(train_sizes, test_scores_mean, 'o-', color="g", label="Cross-validation score")
    plt.legend(loc="best")
    
    return plt

# Import learning_curve
from sklearn.model_selection import learning_curve

# Create a smaller model for learning curve analysis (to save time)
if best_model_name == "LightGBM":
    lc_model = lgb.LGBMRegressor(**{k: v for k, v in best_params.items() if k != 'device_type'})
elif best_model_name == "XGBoost":
    lc_model = XGBRegressor(**{k: v for k, v in xgb_best_params.items() if k not in ['tree_method', 'predictor']})
elif best_model_name == "CatBoost":
    lc_model = CatBoostRegressor(**{k: v for k, v in cat_best_params.items() if k != 'task_type'}, iterations=100, verbose=0)
else:
    # For ensemble models, use LightGBM as a proxy
    lc_model = lgb.LGBMRegressor(**{k: v for k, v in best_params.items() if k != 'device_type'})

# Plot learning curve
plot_learning_curve(lc_model, X_train, y_train, title=f"Learning Curve for {best_model_name}")
plt.show()

# Step 9: Create deployment-ready prediction function
print("\n=== Creating Deployment Function ===")

def predict_eta(new_data, model_path=best_model_path, model_type=best_model_name):
    """
    Makes predictions on new data using the saved model.
    
    Parameters:
    -----------
    new_data : pandas.DataFrame or numpy.array
        New data to make predictions on. Should have the same features as training data.
    model_path : str
        Path to the saved model file.
    model_type : str
        Type of model ('LightGBM', 'XGBoost', 'CatBoost', 'Stacking Ensemble', 'Weighted Voting').
    
    Returns:
    --------
    numpy.array
        Predicted ETA values in minutes.
    """
    # Load the appropriate model
    if model_type == "LightGBM":
        model = lgb.Booster(model_file=model_path)
        return model.predict(new_data)
    elif model_type == "XGBoost":
        model = XGBRegressor()
        model.load_model(model_path)
        return model.predict(new_data)
    elif model_type == "CatBoost":
        model = CatBoostRegressor()
        model.load_model(model_path)
        return model.predict(new_data)
    else:  # Ensemble models
        with open(model_path, 'rb') as file:
            model = pickle.load(file)
        return model.predict(new_data)

# Example usage of the prediction function
print("Prediction function created. Example usage:")
print("predictions = predict_eta(new_data)")

# Step 10: Final
print("\n=== Setting Up Model Monitoring ===")

def create_monitoring_system():
    """
    Creates a function to monitor model performance over time with drift detection
    
    Returns:
    --------
    function
        A function that can be used to log predictions and actuals for performance monitoring
    """
    # Initialize storage for monitoring data
    monitoring_data = {
        'predictions': [],
        'actuals': [],
        'timestamps': [],
        'rmse_over_time': [],
        'mape_over_time': [],
        'drift_detected': []
    }
    
    # Function to log new predictions and actuals
    def log_prediction(prediction, actual, timestamp=None):
        """
        Log a new prediction and its actual value for monitoring
        
        Parameters:
        -----------
        prediction : float
            Predicted ETA value
        actual : float
            Actual observed ETA value
        timestamp : datetime, optional
            Timestamp of the prediction
        """
        if timestamp is None:
            timestamp = pd.Timestamp.now()
        
        monitoring_data['predictions'].append(prediction)
        monitoring_data['actuals'].append(actual)
        monitoring_data['timestamps'].append(timestamp)
        
        # Calculate current performance metrics
        if len(monitoring_data['predictions']) > 10:  # Need enough data
            recent_preds = monitoring_data['predictions'][-100:]  # Last 100 observations
            recent_actuals = monitoring_data['actuals'][-100:]
            
            # Calculate metrics
            rmse = np.sqrt(mean_squared_error(recent_actuals, recent_preds))
            mape = np.mean(np.abs((np.array(recent_actuals) - np.array(recent_preds)) / np.array(recent_actuals))) * 100
            
            monitoring_data['rmse_over_time'].append(rmse)
            monitoring_data['mape_over_time'].append(mape)
            
            # Simple drift detection (standard implementation would be more sophisticated)
            if len(monitoring_data['rmse_over_time']) > 5:
                baseline_rmse = np.mean(monitoring_data['rmse_over_time'][:-5])
                current_rmse = monitoring_data['rmse_over_time'][-1]
                
                # If RMSE increases by more than 20%, flag potential drift
                drift_detected = (current_rmse - baseline_rmse) / baseline_rmse > 0.2
                monitoring_data['drift_detected'].append(drift_detected)
                
                if drift_detected:
                    print(f"WARNING: Potential model drift detected at {timestamp}. Consider retraining.")
    
    # Function to visualize monitoring data
    def visualize_performance():
        """
        Generate visualizations of model performance over time
        """
        if len(monitoring_data['timestamps']) < 10:
            print("Not enough monitoring data collected yet")
            return
        
        # Convert to DataFrame for easier plotting
        df = pd.DataFrame({
            'timestamp': monitoring_data['timestamps'],
            'prediction': monitoring_data['predictions'],
            'actual': monitoring_data['actuals']
        })
        
        # Plot performance over time
        fig, axes = plt.subplots(3, 1, figsize=(15, 18))
        
        # Plot 1: Predictions vs Actuals
        df.set_index('timestamp').plot(y=['prediction', 'actual'], ax=axes[0])
        axes[0].set_title('Predictions vs Actuals Over Time')
        axes[0].set_ylabel('ETA Minutes')
        
        # Plot 2: Error over time
        df['absolute_error'] = np.abs(df['prediction'] - df['actual'])
        df['percentage_error'] = 100 * df['absolute_error'] / df['actual']
        
        df.set_index('timestamp')['absolute_error'].plot(ax=axes[1])
        axes[1].set_title('Absolute Error Over Time')
        axes[1].set_ylabel('Minutes')
        
        # Plot 3: RMSE and MAPE over time
        if len(monitoring_data['rmse_over_time']) > 0:
            rmse_df = pd.DataFrame({
                'timestamp': monitoring_data['timestamps'][-len(monitoring_data['rmse_over_time']):],
                'rmse': monitoring_data['rmse_over_time'],
                'mape': monitoring_data['mape_over_time']
            })
            
            rmse_df.set_index('timestamp').plot(y=['rmse'], ax=axes[2])
            ax2 = axes[2].twinx()
            rmse_df.set_index('timestamp').plot(y=['mape'], ax=ax2, color='red')
            axes[2].set_title('RMSE and MAPE Over Time')
            axes[2].set_ylabel('RMSE (minutes)')
            ax2.set_ylabel('MAPE (%)')
        
        plt.tight_layout()
        plt.show()
        
        # Return the monitoring dataframe for further analysis
        return df
    
    # Return functions for logging and visualization
    return log_prediction, visualize_performance

# Create monitoring functions
log_prediction, visualize_performance = create_monitoring_system()

print("Monitoring system created. Usage example:")
print("log_prediction(predicted_value, actual_value)")
print("visualize_performance()")

# Step 11: Hyperparameter Importance Analysis

print("\n=== Hyperparameter Importance Analysis ===")

def plot_hyperparameter_importance():
    """
    Analyze and visualize the importance of different hyperparameters
    """
    if best_model_name == "LightGBM":
        param_importances = optuna.importance.get_param_importances(study)
    elif best_model_name == "XGBoost":
        param_importances = optuna.importance.get_param_importances(xgb_study)
    elif best_model_name == "CatBoost":
        param_importances = optuna.importance.get_param_importances(cat_study)
    else:
        # For ensemble models, use LightGBM's importance as a proxy
        param_importances = optuna.importance.get_param_importances(study)
    
    # Sort parameters by importance
    param_importances = {k: v for k, v in sorted(param_importances.items(), 
                                                key=lambda item: item[1], reverse=True)}
    
    # Plot importance
    plt.figure(figsize=(10, 6))
    plt.bar(range(len(param_importances)), list(param_importances.values()))
    plt.xticks(range(len(param_importances)), list(param_importances.keys()), rotation=90)
    plt.title(f'Hyperparameter Importance for {best_model_name}')
    plt.tight_layout()
    plt.show()
    
    return param_importances

# Plot hyperparameter importance
hyperparam_importance = plot_hyperparameter_importance()
print("Top 5 most important hyperparameters:")
for i, (param, importance) in enumerate(list(hyperparam_importance.items())[:5]):
    print(f"{i+1}. {param}: {importance:.4f}")

# Step 12: Setup Final Prediction Pipeline

print("\n=== Creating Production Prediction Pipeline ===")

from sklearn.pipeline import Pipeline

def create_prediction_pipeline():
    """
    Creates a complete prediction pipeline that can be used in production
    
    Returns:
    --------
    Pipeline
        A scikit-learn pipeline for making predictions
    """
    # Load the best model
    if best_model_name == "LightGBM":
        model = lgb.LGBMRegressor()
        model.booster_ = lgb.Booster(model_file=best_model_path)
    elif best_model_name == "XGBoost":
        model = XGBRegressor()
        model.load_model(best_model_path)
    elif best_model_name == "CatBoost":
        model = CatBoostRegressor()
        model.load_model(best_model_path)
    else:  # Ensemble models
        with open(best_model_path, 'rb') as file:
            model = pickle.load(file)
    
    # Create pipeline
    pipeline = Pipeline([
        ('model', model)
    ])
    
    # Save the pipeline
    with open('final_eta_prediction_pipeline.pkl', 'wb') as f:
        pickle.dump(pipeline, f)
    
    print(f"Final prediction pipeline saved as 'final_eta_prediction_pipeline.pkl'")
    
    # Define prediction function
    def predict_with_pipeline(X):
        """
        Make predictions using the pipeline
        
        Parameters:
        -----------
        X : pandas.DataFrame
            Features for prediction
            
        Returns:
        --------
        numpy.ndarray
            Predicted ETA values
        """
        return pipeline.predict(X)
    
    return predict_with_pipeline

# Create prediction pipeline
predict_with_pipeline = create_prediction_pipeline()

print("Production prediction pipeline created. Usage example:")
print("predictions = predict_with_pipeline(new_data)")

# Step 13: Conclusion and Final Results Summary

print("\n=== Final Results Summary ===")

# Create summary table
summary_data = []
for model_name, results in models_comparison.items():
    summary_data.append({
        'Model': model_name,
        'RMSE': results['rmse'],
        'MAE': results['mae'],
        'R²': results['r2'],
        'MAPE (%)': results['mape'],
        'Accuracy (%)': results['accuracy']
    })

summary_df = pd.DataFrame(summary_data)
print(summary_df.sort_values('RMSE'))

# Print best model details
print(f"\nBest model: {best_model_name}")
print(f"RMSE: {models_comparison[best_model_name]['rmse']:.4f}")
print(f"Prediction Accuracy: {models_comparison[best_model_name]['accuracy']:.2f}%")

# Feature importance summary
print("\nTop 5 most important features:")
if best_model_name == "LightGBM":
    importance = dict(zip(X_train.columns, lgb_optimized.feature_importance()))
elif best_model_name == "XGBoost":
    importance = xgb_model.get_booster().get_score(importance_type='gain')
elif best_model_name == "CatBoost":
    importance = dict(zip(X_train.columns, cat_model.get_feature_importance()))
else:
    # For ensemble models, use LightGBM's importance as a proxy
    importance = dict(zip(X_train.columns, lgb_optimized.feature_importance()))

# Sort importance
importance = {k: v for k, v in sorted(importance.items(), key=lambda item: item[1], reverse=True)}
for i, (feature, imp) in enumerate(list(importance.items())[:5]):
    print(f"{i+1}. {feature}: {imp}")

print("\nOptimization complete! The model is ready for deployment.")

c:\Users\cheng\anaconda3\envs\GPU\lib\site-packages\tqdm\auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2025-03-18 04:08:45,275] A new study created in memory with name: no-name-f2e15ebb-d300-4b6e-bf18-1520b779007b


Data shape: (100000, 14)
Training set shape: (80000, 12)
Testing set shape: (20000, 12)

=== Optimizing LightGBM with Optuna ===


[I 2025-03-18 04:08:46,818] Trial 0 finished with value: 0.24878254696984378 and parameters: {'boosting_type': 'gbdt', 'num_leaves': 302, 'learning_rate': 0.03945110809150444, 'feature_fraction': 0.6519721605489356, 'min_child_samples': 67, 'max_depth': 4, 'lambda_l1': 0.0017971489835258824, 'lambda_l2': 0.0004780592075506056, 'min_split_gain': 3.0653382881307025, 'bagging_fraction': 0.7435680636621519, 'bagging_freq': 8}. Best is trial 0 with value: 0.24878254696984378.
[W 2025-03-18 04:10:00,112] Trial 1 failed because of the following error: KeyboardInterrupt()
Traceback (most recent call last):
  File "c:\Users\cheng\anaconda3\envs\GPU\lib\site-packages\optuna\study\_optimize.py", line 196, in _run_trial
    value_or_values = func(trial)
  File "<ipython-input-7-8ad47805d9af>", line 155, in objective
    callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)]
  File "c:\Users\cheng\anaconda3\envs\GPU\lib\site-packages\lightgbm\engine.py", line 276, in train
    booster.u

KeyboardInterrupt: 